In [ ]:
!pip install -q groq openai pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import time
import random
import xml.etree.ElementTree as ET
from typing import Optional

import pandas as pd
from groq import Groq
import openai
from google.colab import userdata, drive


In [ ]:
# --- Conectar ao Google Drive (necessário para salvar/ler o checkpoint) ---
drive.mount('/content/drive')

# Pasta no Drive onde checkpoint e resultados serão salvos.
# Ajuste se quiser organizar em outra pasta.
PASTA_PROJETO = "/content/drive/MyDrive/PIBIC_grafos_conhecimento"
os.makedirs(PASTA_PROJETO, exist_ok=True)

CHECKPOINT_PATH = os.path.join(PASTA_PROJETO, "checkpoint_resultados.csv")
CHECKPOINT_JUIZ_PATH = os.path.join(PASTA_PROJETO, "checkpoint_avaliacao_juiz.csv")

print(f"Pasta do projeto: {PASTA_PROJETO}")
print(f"Checkpoint de extração: {CHECKPOINT_PATH}")
print(f"Checkpoint de avaliação do juiz: {CHECKPOINT_JUIZ_PATH}")


Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/PIBIC_grafos_conhecimento
Checkpoint de extração: /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_resultados.csv
Checkpoint de avaliação do juiz: /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_avaliacao_juiz.csv


In [ ]:
# --- Conectar à planilha do Google Sheets (abas "Modelos" e "Judge") ---
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default as _google_auth_default
_creds, _ = _google_auth_default()
gc = gspread.authorize(_creds)

PLANILHA_RESULTADOS_ID = userdata.get('PLANILHA_RESULTADOS_ID')
planilha_resultados = gc.open_by_key(PLANILHA_RESULTADOS_ID)
aba_modelos = planilha_resultados.worksheet("Modelos")
aba_judge = planilha_resultados.worksheet("Judge")

print(f"Conectado à planilha: {planilha_resultados.title}")

Conectado à planilha: Resultados - Emmanuel


In [ ]:
import math

COLUNAS_MODELOS_SHEET = ["drug", "categoria", "texto_fonte", "modelo", "estrategia", "prompt", "resposta_bruta", "triplas", "n_triplas", "parsing_ok", "predicado_categoria_ok", "tokens_prompt", "tokens_resposta", "latencia_segundos", "erro"]

COLUNAS_JUDGE_SHEET = ["drug", "categoria", "texto_fonte", "modelo", "estrategia", "triplas", "n_triplas", "parsing_ok", "nota_format", "nota_completeness", "nota_correctness", "nota_no_hallucination", "justification"]

def _valor_para_sheet(valor):
    return json.dumps(valor, ensure_ascii=False) if isinstance(valor, list) else ("" if (valor is None or (isinstance(valor, float) and math.isnan(valor))) else valor)

def enviar_linhas_para_sheet(worksheet, linhas, colunas):
  valores = [[_valor_para_sheet(linha.get(col)) for col in colunas] for linha in linhas] if linhas else []
  if valores:
    worksheet.append_rows(valores, value_input_option="USER_ENTERED")

In [ ]:
# --- Clientes de API ---

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Cliente Maritaca - usado APENAS como juiz (nao como candidato)
maritaca_client = openai.OpenAI(api_key=userdata.get('Maritaca'), base_url="https://chat.maritaca.ai/api")

# --- Definicao dos modelos candidatos (2 familias distintas, via Groq) ---
# gpt-oss-120b -> familia OpenAI (via Groq)
# qwen-3.6-27b -> familia Alibaba/Qwen (via Groq)
MODELOS = {"gpt-oss-120b": {"client": "groq", "model_id": "openai/gpt-oss-120b"}, "qwen-3.6-27b": {"client": "groq", "model_id": "qwen/qwen3.6-27b", "reasoning_effort": "none"}, "llama-3.3-70b": {"client": "groq", "model_id": "llama-3.3-70b-versatile"}}
# --- Juiz: Maritaca sabia-4 (familia diferente dos candidatos, elimina self-preference bias) ---
JUIZ_CLIENT = "maritaca"
JUIZ_MODELO_ID = "sabia-4"

In [ ]:
CAMINHO_XML_DRIVE = os.path.join(PASTA_PROJETO, "metabolism_absorption.xml")

if not os.path.exists(CAMINHO_XML_DRIVE):
    print("Arquivo XML não encontrado no Drive. Faça upload uma única vez:")
    from google.colab import files
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

    import shutil
    shutil.copy(file_name, CAMINHO_XML_DRIVE)
    print(f"Arquivo salvo permanentemente em: {CAMINHO_XML_DRIVE}")
else:
    print(f"Arquivo XML já encontrado no Drive: {CAMINHO_XML_DRIVE}")

tree = ET.parse(CAMINHO_XML_DRIVE)
root = tree.getroot()
print(f"Total de elementos <drug> no XML: {len(root.findall('drug'))}")


Arquivo XML já encontrado no Drive: /content/drive/MyDrive/PIBIC_grafos_conhecimento/metabolism_absorption.xml
Total de elementos <drug> no XML: 2628


In [ ]:
MIN_CHARS_TEXTO = 30  # textos com menos que isso são descartados (ex: "Hepatic")

def limpar_texto(texto: Optional[str]) -> Optional[str]:
    """
    Remove ruído típico do dataset (DrugBank):
    - referências bibliográficas tipo [L41539], [A246609,A227963]
    - tags HTML residuais (<sub>, <sup>, etc.)
    - espaços/quebras de linha excessivos

    Retorna None se o texto limpo tiver menos de MIN_CHARS_TEXTO caracteres
    (filtra textos incompletos como "Hepatic" que causam alucinação nos modelos).
    """
    if not texto:
        return None
    texto = re.sub(r"\[[A-Z]\d+(?:,\s*[A-Z]\d+)*\]", "", texto)
    texto = re.sub(r"<[^>]+>", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    if len(texto) < MIN_CHARS_TEXTO:
        return None  # descarta textos muito curtos
    return texto


In [ ]:
registros = []
for drug in root.findall("drug"):
    name = drug.find("name").text if drug.find("name") is not None else "N/A"
    metabolism = drug.find("metabolism").text if drug.find("metabolism") is not None else None
    absorption = drug.find("absorption").text if drug.find("absorption") is not None else None

    metabolism = limpar_texto(metabolism)
    absorption = limpar_texto(absorption)

    if metabolism:
        registros.append({"drug": name, "categoria": "Metabolismo", "texto": metabolism})
    if absorption:
        registros.append({"drug": name, "categoria": "Absorção", "texto": absorption})

df_textos = pd.DataFrame(registros)
print(f"Total de textos carregados (após limpeza e filtro de textos curtos): {len(df_textos)}")
df_textos.head()

df_textos["has_bioavailability"] = df_textos["texto"].str.contains("bioavailability", case=False, na=False)
df_textos["has_metabolism"] = df_textos["categoria"] == "Metabolismo"

n_bioavailability = df_textos["has_bioavailability"].sum()
print(f"Textos que mencionam bioavailability: {n_bioavailability} de {len(df_textos)}")


Total de textos carregados (após limpeza e filtro de textos curtos): 4327
Textos que mencionam bioavailability: 825 de 4327


In [ ]:
TRIPLA_FIXA_BIOAVAILABILITY = "[DRUG]::bioavailability::56.9%"
# Palavras-chave usadas na validacao pos-parsing (validar_predicado_categoria)
# para confirmar que ao menos uma tripla reflete a categoria do texto fonte.
PALAVRAS_CHAVE_METABOLISMO = [
    "metaboli", "hydrolyz", "hydrolys", "biotransform", "cytochrome",
    "cyp", "enzyme", "metabolite", "conjugat", "oxidiz", "oxidat",
]
PALAVRAS_CHAVE_ABSORCAO = [
    "absorb", "absorption", "bioavailab", "tmax", "release", "uptake",
]

def montar_prompt_zero_shot(drug: str, categoria: str, texto: str) -> str:
    return f"""Task: Extract knowledge triples (Subject, Predicate, Object) from the text below, paying special attention to bioavailability, metabolism, and absorption information when present.
Drug: {drug}
Category: {categoria}
Text: {texto}
Instructions:
- Reply ONLY with the triples, one per line, in the format: subject::predicate::object
- Use ONLY information explicitly present in the text. Do NOT infer the drug name if it is not in the text.
- Use exactly "::" (double colon) to separate subject, predicate, and object.
- Do NOT use JSON, bullet points, or any text before or after the triples.
- ALL output MUST be in English. Do not use Portuguese or any other language.
- If the text mentions bioavailability (e.g., a percentage or value), you MUST include a triple with predicate "bioavailability", for example: {TRIPLA_FIXA_BIOAVAILABILITY}.
- If the text mentions metabolism (e.g., enzymes, metabolic pathways, or metabolites involved), you MUST include a triple with predicate "metabolism", for example: [DRUG]::metabolism::hepatic cytochrome P450 enzymes.
- This text belongs to Category "{categoria}". Regardless of the rules above, you MUST include AT LEAST ONE triple whose predicate is directly related to this category, using only information explicitly supported by the text:
  - If Category is "Metabolismo", use a predicate such as "metabolism", "is metabolized to", "is hydrolyzed to", "is biotransformed into", or an equivalent metabolic relation.
  - If Category is "Absorção", use a predicate such as "absorption", "is absorbed via", "bioavailability", "has Tmax of", or an equivalent absorption relation.
Example output format:
X::Y::Z
X::W::V
Answer:"""

def montar_prompt_few_shot(drug: str, categoria: str, texto: str) -> str:
    return f"""Task: Extract knowledge triples (Subject, Predicate, Object) from pharmacological texts, paying special attention to bioavailability, metabolism, and absorption information when present.
ALL output MUST be in English. Do not use Portuguese or any other language.
Example 1:
Text: As a polypeptide, it is metabolized by the sequential cleavage of amino acids by kidney exoproteases.
Answer:
[DRUG]::metabolized::kidney exoproteases
[DRUG]::undergoes::sequential cleavage of amino acids
Example 2:
Text: Subcutaneous bioavailability is 56.9% with a Tmax of 69h.
Answer:
{TRIPLA_FIXA_BIOAVAILABILITY}
[DRUG]::has Tmax of::69h
Now extract triples for:
Drug: {drug}
Category: {categoria}
Text: {texto}
Instructions:
- Reply ONLY with the triples, one per line, in the format: subject::predicate::object
- Use ONLY information explicitly present in the text. Do NOT infer the drug name if it is not in the text.
- Use exactly "::" to separate the fields.
- Do NOT use JSON, bullet points, or any text before or after the triples.
- ALL output MUST be in English. Do not use Portuguese or any other language.
- If the text mentions bioavailability (e.g., a percentage or value), you MUST include a triple with predicate "bioavailability", for example: {TRIPLA_FIXA_BIOAVAILABILITY}.
- If the text mentions metabolism (e.g., enzymes, metabolic pathways, or metabolites involved), you MUST include a triple with predicate "metabolism", for example: [DRUG]::metabolism::hepatic cytochrome P450 enzymes.
- This text belongs to Category "{categoria}". Regardless of the rules above, you MUST include AT LEAST ONE triple whose predicate is directly related to this category, using only information explicitly supported by the text:
  - If Category is "Metabolismo", use a predicate such as "metabolism", "is metabolized to", "is hydrolyzed to", "is biotransformed into", or an equivalent metabolic relation.
  - If Category is "Absorção", use a predicate such as "absorption", "is absorbed via", "bioavailability", "has Tmax of", or an equivalent absorption relation.
Answer:"""

def validar_predicado_categoria(categoria: str, triplas: Optional[list]) -> bool:
    """
    Verifica se ao menos uma tripla tem predicado correlato a categoria do
    texto fonte (Metabolismo -> palavras-chave de metabolismo; Absorcao ->
    palavras-chave de absorcao). E uma checagem semantica complementar ao
    parsing_ok, que so garante o formato sujeito::predicado::objeto, nao o
    conteudo esperado.
    """
    if not triplas:
        return False
    palavras_chave = (
        PALAVRAS_CHAVE_METABOLISMO if categoria == "Metabolismo" else PALAVRAS_CHAVE_ABSORCAO
    )
    for tripla in triplas:
        predicado = str(tripla.get("predicado", "")).lower()
        if any(chave in predicado for chave in palavras_chave):
            return True
    return False

s = montar_prompt_zero_shot('d','c','t')
print(len(s))
print('few_shot' in s)
print(repr(s[-80:]))

1638
False
'n equivalent absorption relation.\nExample output format:\nX::Y::Z\nX::W::V\nAnswer:'


In [ ]:
s = montar_prompt_zero_shot('d','c','t')
print(len(s))
print('few_shot' in s)
print(repr(s[-80:]))

1638
False
'n equivalent absorption relation.\nExample output format:\nX::Y::Z\nX::W::V\nAnswer:'


In [ ]:
import threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed

class CotaEsgotadaError(Exception):
    """Levantada quando a API retorna 429 (limite de requisicoes/tokens atingido)."""
    pass

def eh_erro_rate_limit(excecao: Exception) -> bool:
    """Detecta se a excecao corresponde a um erro 429 (rate limit / cota esgotada)."""
    texto_erro = str(excecao).lower()
    return "429" in texto_erro or "rate limit" in texto_erro or "rate_limit" in texto_erro


class RateLimiter:
    """
    Limitador de taxa thread-safe baseado em janela deslizante de 60s.

    Permite paralelizar chamadas de API com seguranca: varias threads podem
    chamar aguardar() ao mesmo tempo, e o limiter garante que no maximo
    max_por_minuto chamadas sejam liberadas em qualquer janela de 60s,
    bloqueando (time.sleep) as chamadas excedentes ate haver espaco.

    Se max_por_minuto for None, nunca bloqueia (usado para clientes sem
    limite restritivo conhecido, como Groq Developer Tier).
    """
    def __init__(self, max_por_minuto: Optional[int] = None):
        self.max_por_minuto = max_por_minuto
        self.chamadas = deque()
        self.lock = threading.Lock()

    def aguardar(self):
        if self.max_por_minuto is None:
            return
        while True:
            with self.lock:
                agora = time.time()
                while self.chamadas and agora - self.chamadas[0] > 60:
                    self.chamadas.popleft()
                if len(self.chamadas) < self.max_por_minuto:
                    self.chamadas.append(agora)
                    return
                espera = 60 - (agora - self.chamadas[0]) + 0.05
            time.sleep(max(espera, 0.05))


RATE_LIMITERS = {
    "groq": RateLimiter(max_por_minuto=None),
    "openrouter": RateLimiter(max_por_minuto=18),
    "maritaca": RateLimiter(max_por_minuto=None),
}

def chamar_modelo(nome_modelo: str, prompt: str, max_tokens: int = 2000,
                   max_tentativas: int = 3) -> dict:
    """
    Chama o modelo identificado por nome_modelo (chave em MODELOS) com o prompt dado.
    Retorna dict com: texto_resposta, tokens_prompt, tokens_resposta, latencia_segundos.
    Antes de cada tentativa, aguarda o RateLimiter do cliente correspondente
    (importante ao chamar esta funcao de varias threads em paralelo).
    Se a cota/limite de taxa for atingido (429), levanta CotaEsgotadaError.
    Outros erros sao re-tentados com backoff exponencial.
    """
    config = MODELOS[nome_modelo]
    limiter = RATE_LIMITERS.get(config["client"])

    for tentativa in range(1, max_tentativas + 1):
        if limiter is not None:
            limiter.aguardar()
        inicio = time.time()
        try:
            if config["client"] == "groq":
                kwargs_extra = {"reasoning_effort": config["reasoning_effort"]} if "reasoning_effort" in config else {}
                resp = groq_client.chat.completions.create(
                    model=config["model_id"],
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens,
                    temperature=0.0,
                    **kwargs_extra,
                )

            elif config["client"] == "openrouter":
                resp = openrouter_client.chat.completions.create(
                    model=config["model_id"],
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens,
                    temperature=0.0,
                    extra_headers={
                        "HTTP-Referer": "https://github.com/pibic-grafos-conhecimento",
                        "X-Title": "PIBIC Grafos de Conhecimento",
                    },
                )
            elif config["client"] == "maritaca":
                resp = maritaca_client.chat.completions.create(
                    model=config["model_id"],
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens,
                    temperature=0.0,
                )

            else:
                raise ValueError(f"Client desconhecido para modelo {nome_modelo}")

            texto = resp.choices[0].message.content
            usage = getattr(resp, "usage", None)
            tokens_prompt = getattr(usage, "prompt_tokens", None) if usage else None
            tokens_resposta = getattr(usage, "completion_tokens", None) if usage else None
            latencia = round(time.time() - inicio, 3)
            return {
                "texto_resposta": texto,
                "tokens_prompt": tokens_prompt,
                "tokens_resposta": tokens_resposta,
                "latencia_segundos": latencia,
            }

        except Exception as e:
            if eh_erro_rate_limit(e):
                raise CotaEsgotadaError(f"Cota esgotada para {nome_modelo}: {e}") from e

            if tentativa == max_tentativas:
                raise

            espera = 2 ** tentativa
            print(f"   Erro transitorio em {nome_modelo} (tentativa {tentativa}/{max_tentativas}): {e}")
            print(f"   Aguardando {espera}s antes de tentar novamente...")
            time.sleep(espera)

def extrair_triplas_delimitadas(texto_resposta: str) -> Optional[list]:
    """
    Extrai triplas no formato "sujeito::predicado::objeto", uma por linha.
    Remove antes do parsing qualquer raciocinio vazado pelo modelo (ex.:
    blocos <think>...</think>/<reasoning>...</reasoning>, inclusive quando
    a tag de abertura aparece sem fechamento por causa de truncamento).
    Linhas sem o delimitador sao ignoradas silenciosamente.
    """
    if not texto_resposta:
        return None

    padrao_raciocinio = re.compile(
        r"<\s*(think|thinking|reasoning|racioc[ií]nio)\s*>.*?<\s*/\s*\1\s*>",
        re.IGNORECASE | re.DOTALL,
    )
    padrao_raciocinio_aberto = re.compile(
        r"<\s*(think|thinking|reasoning|racioc[ií]nio)\s*>.*$",
        re.IGNORECASE | re.DOTALL,
    )

    texto_limpo = padrao_raciocinio.sub("", texto_resposta)
    texto_limpo = padrao_raciocinio_aberto.sub("", texto_limpo)
    texto_limpo = re.sub(r"```(?:\w+)?", "", texto_limpo).strip()

    triplas = []
    for linha in texto_limpo.split("\n"):
        linha = linha.strip()
        if not linha or "::" not in linha:
            continue

        linha = re.sub(r"^[\-\*\d\.\)]+\s*", "", linha)

        campos = linha.split("::")
        if len(campos) != 3:
            continue

        sujeito, predicado, objeto = (c.strip() for c in campos)
        if not sujeito or not predicado or not objeto:
            continue

        triplas.append({"sujeito": sujeito, "predicado": predicado, "objeto": objeto})

    return triplas if triplas else None

def triplas_para_texto(triplas: list) -> str:
    """Converte uma lista de triplas (dicts) de volta para o formato delimitado."""
    return "\n".join(f"{t['sujeito']}::{t['predicado']}::{t['objeto']}" for t in triplas)

In [ ]:
COLUNAS_RESULTADO = [
    "drug", "categoria", "texto_fonte", "modelo", "estrategia", "prompt",
    "resposta_bruta", "triplas", "n_triplas", "parsing_ok",
        "predicado_categoria_ok",
    "tokens_prompt", "tokens_resposta", "latencia_segundos", "erro",
]

def carregar_checkpoint(caminho: str) -> pd.DataFrame:
    if os.path.exists(caminho):
        df = pd.read_csv(caminho)
        # A coluna "triplas" foi salva como string JSON; convertemos de volta para lista/None
        if "triplas" in df.columns:
            df["triplas"] = df["triplas"].apply(
                lambda x: json.loads(x) if isinstance(x, str) and x.strip().startswith("[") else None
            )
        print(f"Checkpoint carregado: {len(df)} execuções já registradas em {caminho}")
        return df
    print(f"Nenhum checkpoint encontrado em {caminho}. Começando do zero.")
    return pd.DataFrame(columns=COLUNAS_RESULTADO)

def salvar_checkpoint(df: pd.DataFrame, caminho: str):
    df_salvar = df.copy()
    if "triplas" in df_salvar.columns:
        df_salvar["triplas"] = df_salvar["triplas"].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if x is not None else None
        )
    df_salvar.to_csv(caminho, index=False)

def chave_execucao(drug, categoria, modelo, estrategia):
    return (drug, categoria, modelo, estrategia)

def _concat_novos(df_base: pd.DataFrame, novos: list) -> pd.DataFrame:
    """
    Concatena novos (lista de dicts) a df_base, evitando o FutureWarning do
    pandas ao concatenar com DataFrames vazios/all-NA (comum quando df_base
    ainda nao tem nenhuma linha, no inicio de uma sessao nova).
    """
    if not novos:
        return df_base.copy()
    df_novos = pd.DataFrame(novos)
    if df_base is None or len(df_base) == 0:
        return df_novos
    return pd.concat([df_base, df_novos], ignore_index=True)

In [ ]:
PROMPT_JUIZ_TEMPLATE = """You are an expert evaluator in information extraction and knowledge graph construction.

Evaluate the quality of the triples (subject, predicate, object) extracted from a pharmacological text.

SOURCE TEXT:
{texto_fonte}

EXTRACTED TRIPLES (format subject::predicate::object, one per line):
{triplas_formatadas}

Evaluate the following criteria, assigning a score from 1 (very poor) to 5 (excellent) for each:

1. format: do the triples follow a valid and coherent subject-predicate-object structure?
5 = all triples are well-formed and semantically coherent (no empty, truncated, or malformed fields).
4 = almost all triples are well-formed; one minor issue but still understandable.
3 = half or more of the triples are well-formed, but with noticeable errors (inverted fields, vague predicate, incomplete object).
2 = most triples have structural problems (missing fields, mixed subject/object, meaningless predicates).
1 = triples have essentially no valid structure or are unreadable.

2. completeness: do the triples capture the important relations present in the source text?
5 = all relevant relations in the source text were captured.
4 = almost all important relations were captured; only a minor secondary detail is missing.
3 = the main relations were captured, but relevant secondary relations are missing.
2 = only a small part of the relations in the text was captured.
1 = the extraction ignores almost all relations present in the text.

3. correctness: do the triples faithfully reflect what is written in the source text?
5 = all triples exactly reflect what is written in the text (no distortion of meaning).
4 = almost all triples are faithful; there is one slight inaccuracy (e.g., paraphrased without changing meaning).
3 = most triples are faithful, but one or more show noticeable distortion of meaning.
2 = several triples distort or invert the meaning of the text.
1 = the triples seriously contradict or distort what the text states.

4. no_hallucination: do the triples NOT invent information absent from the text?
5 = nothing invented; everything is explicitly in the text.
4 = almost nothing invented; at most one very small and reasonable inference.
3 = one triple contains information clearly not present in the text.
2 = several triples contain invented or unsupported information.
1 = most of the extracted content was invented, without basis in the text.

Reply ONLY in JSON format, without any text before or after, following exactly this structure:
{{"format": <score>, "completeness": <score>, "correctness": <score>, "no_hallucination": <score>, "justification": "<brief justification in 1-2 sentences>"}}
"""

def chamar_juiz(texto_fonte: str, triplas: list, max_tentativas: int = 3) -> dict:
    prompt = PROMPT_JUIZ_TEMPLATE.format(
        texto_fonte=texto_fonte,
        triplas_formatadas=triplas_para_texto(triplas),
    )

    limiter = RATE_LIMITERS.get(JUIZ_CLIENT)

    for tentativa in range(1, max_tentativas + 1):
        if limiter is not None:
            limiter.aguardar()
        try:
            # Juiz: Maritaca sabia-4 (família distinta dos candidatos)
            resp = maritaca_client.chat.completions.create(
                model=JUIZ_MODELO_ID,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=800,
                temperature=0.0,
            )
            texto = resp.choices[0].message.content

            texto_limpo = re.sub(r"```(?:json)?", "", texto).strip()
            match = re.search(r"\{.*\}", texto_limpo, re.DOTALL)
            candidato = match.group(0) if match else texto_limpo

            return json.loads(candidato)

        except Exception as e:
            if eh_erro_rate_limit(e):
                raise CotaEsgotadaError(f"Cota esgotada para o juiz ({JUIZ_MODELO_ID}): {e}") from e
            if tentativa == max_tentativas:
                raise
            time.sleep(2 ** tentativa)

COLUNAS_AVALIACAO = [
    "drug", "categoria", "modelo", "estrategia",
    "nota_format", "nota_completeness", "nota_correctness",
    "nota_no_hallucination",
    "justification", "erro_juiz",
    ]


def avaliar_com_juiz_checkpoint(
    df_resultados: pd.DataFrame,
    caminho_checkpoint_juiz: str,
    delay_entre_chamadas: float = 0.1,
    salvar_a_cada: int = 20,
    max_workers: int = 8,
) -> pd.DataFrame:

    if os.path.exists(caminho_checkpoint_juiz):
        df_avaliacoes = pd.read_csv(caminho_checkpoint_juiz)
        print(f"Checkpoint do juiz carregado: {len(df_avaliacoes)} avaliações já feitas.")
    else:
        df_avaliacoes = pd.DataFrame(columns=COLUNAS_AVALIACAO)
        print("Nenhum checkpoint do juiz encontrado. Começando do zero.")

    chaves_avaliadas = set(
        zip(df_avaliacoes["drug"], df_avaliacoes["categoria"],
            df_avaliacoes["modelo"], df_avaliacoes["estrategia"])
    ) if len(df_avaliacoes) > 0 else set()

    elegiveis = df_resultados[df_resultados["parsing_ok"]].copy()

    # Monta a lista de tarefas pendentes (sequencial e leve, so organiza os dados)
    tarefas = []
    for _, row in elegiveis.iterrows():
        chave = (row["drug"], row["categoria"], row["modelo"], row["estrategia"])
        if chave in chaves_avaliadas:
            continue
        triplas_row = row["triplas"]
        if isinstance(triplas_row, str):
            try:
                triplas_row = json.loads(triplas_row)
            except Exception:
                triplas_row = None
        if not triplas_row:
            continue
        tarefas.append({"row": row, "triplas": triplas_row})

    def _avaliar_tarefa(tarefa):
        """Roda em worker thread: so a chamada de API (I/O), sem tocar em estado compartilhado."""
        try:
            nota = chamar_juiz(tarefa["row"]["texto_fonte"], tarefa["triplas"])
            return {"tarefa": tarefa, "nota": nota, "erro_juiz": None, "esgotado": False}
        except CotaEsgotadaError:
            return {"tarefa": tarefa, "nota": None, "erro_juiz": None, "esgotado": True}
        except Exception as e:
            nota_vazia = {"format": None, "completeness": None, "correctness": None,
                          "no_hallucination": None, "justification": None}
            return {"tarefa": tarefa, "nota": nota_vazia, "erro_juiz": str(e), "esgotado": False}

    novas_avaliacoes = []
    contador = 0
    juiz_esgotado = False
    tamanho_bloco = max_workers * 5

    idx = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while idx < len(tarefas) and not juiz_esgotado:
            bloco = tarefas[idx: idx + tamanho_bloco]
            idx += len(bloco)

            futuros = [executor.submit(_avaliar_tarefa, t) for t in bloco]

            for futuro in as_completed(futuros):
                resultado = futuro.result()

                if resultado["esgotado"]:
                    if not juiz_esgotado:
                        print("⚠️ Cota do juiz esgotada nesta sessão. Salvando progresso e parando.")
                    juiz_esgotado = True
                    continue

                row = resultado["tarefa"]["row"]
                nota = resultado["nota"]
                novas_avaliacoes.append({
                    "drug": row["drug"],
                    "categoria": row["categoria"],
                    "modelo": row["modelo"],
                    "estrategia": row["estrategia"],
                    "nota_format": nota.get("format"),
                    "nota_completeness": nota.get("completeness"),
                    "nota_correctness": nota.get("correctness"),
                    "nota_no_hallucination": nota.get("no_hallucination"),
                    "justification": nota.get("justification"),
                    "erro_juiz": resultado["erro_juiz"],
                })
                contador += 1

                if contador % 10 == 0:
                    print(f"[{contador} avaliações nesta sessão] {row['modelo']} | {row['estrategia']} | {row['drug']}")

                if contador % salvar_a_cada == 0:
                    df_atual = _concat_novos(df_avaliacoes, novas_avaliacoes)
                    df_atual.to_csv(caminho_checkpoint_juiz, index=False)

    df_final = _concat_novos(df_avaliacoes, novas_avaliacoes)
    df_final.to_csv(caminho_checkpoint_juiz, index=False)
    print(f"\n✅ {contador} novas avaliações nesta sessão. Total acumulado: {len(df_final)} / {len(elegiveis)}")
    if juiz_esgotado:
        print("Rode esta célula novamente para continuar de onde parou.")

    return df_final

In [ ]:
COLUNAS_AVALIACAO = [
    "drug", "categoria", "modelo", "estrategia",
    "nota_format", "nota_completeness", "nota_correctness",
    "nota_no_hallucination", "justification", "erro_juiz",
]


def avaliar_com_juiz_checkpoint(
    df_resultados: pd.DataFrame,
    caminho_checkpoint_juiz: str,
    delay_entre_chamadas: float = 0.1,
    salvar_a_cada: int = 20,
    enviar_para_sheets: bool = False,
) -> pd.DataFrame:

    if os.path.exists(caminho_checkpoint_juiz):
        df_avaliacoes = pd.read_csv(caminho_checkpoint_juiz)
        print(f"Checkpoint do juiz carregado: {len(df_avaliacoes)} avaliações já feitas.")
    else:
        df_avaliacoes = pd.DataFrame(columns=COLUNAS_AVALIACAO)
        print("Nenhum checkpoint do juiz encontrado. Começando do zero.")

    chaves_avaliadas = set(
        zip(df_avaliacoes["drug"], df_avaliacoes["categoria"],
            df_avaliacoes["modelo"], df_avaliacoes["estrategia"])
    ) if len(df_avaliacoes) > 0 else set()

    elegiveis = df_resultados[df_resultados["parsing_ok"]].copy()

    novas_avaliacoes = []
    buffer_sheet_judge = []
    contador = 0
    juiz_esgotado = False

    for _, row in elegiveis.iterrows():
        chave = (row["drug"], row["categoria"], row["modelo"], row["estrategia"])
        if chave in chaves_avaliadas:
            continue

        if juiz_esgotado:
            break

        # Garantir que triplas é lista (pode vir como string do CSV)
        triplas_row = row["triplas"]
        if isinstance(triplas_row, str):
            try:
                import json as _json
                triplas_row = _json.loads(triplas_row)
            except Exception:
                triplas_row = None
        if not triplas_row:
            continue  # sem triplas válidas, pula

        try:
            nota = chamar_juiz(row["texto_fonte"], triplas_row)
            erro_juiz = None
        except CotaEsgotadaError:
            print("⚠️  Cota do juiz esgotada nesta sessão. Salvando progresso e parando.")
            juiz_esgotado = True
            break
        except Exception as e:
            nota = {"format": None, "completeness": None, "correctness": None,
                     "no_hallucination": None, "bioavailability": None, "justification": None}
            erro_juiz = str(e)

        novas_avaliacoes.append({
            "drug": row["drug"],
            "categoria": row["categoria"],
            "modelo": row["modelo"],
            "estrategia": row["estrategia"],
            "nota_format": nota.get("format"),
            "nota_completeness": nota.get("completeness"),
            "nota_correctness": nota.get("correctness"),
            "nota_no_hallucination": nota.get("no_hallucination"),
            "nota_bioavailability": nota.get("bioavailability"),
            "justification": nota.get("justification"),
            "erro_juiz": erro_juiz,
        })
        buffer_sheet_judge.append({"drug": row["drug"], "categoria": row["categoria"], "texto_fonte": row["texto_fonte"], "modelo": row["modelo"], "estrategia": row["estrategia"], "triplas": triplas_row, "n_triplas": row["n_triplas"], "parsing_ok": row["parsing_ok"], "nota_format": nota.get("format"), "nota_completeness": nota.get("completeness"), "nota_correctness": nota.get("correctness"), "nota_no_hallucination": nota.get("no_hallucination"), "justification": nota.get("justification")})
        chaves_avaliadas.add(chave)
        contador += 1

        if contador % 10 == 0:
            print(f"[{contador} avaliações nesta sessão] {row['modelo']} | {row['estrategia']} | {row['drug']}")

        if contador % salvar_a_cada == 0:
            df_atual = pd.concat([df_avaliacoes, pd.DataFrame(novas_avaliacoes)], ignore_index=True)
            df_atual.to_csv(caminho_checkpoint_juiz, index=False)
            if enviar_para_sheets:             enviar_linhas_para_sheet(aba_judge, buffer_sheet_judge, COLUNAS_JUDGE_SHEET)
            buffer_sheet_judge = []

        time.sleep(delay_entre_chamadas)

    df_final = pd.concat([df_avaliacoes, pd.DataFrame(novas_avaliacoes)], ignore_index=True)
    df_final.to_csv(caminho_checkpoint_juiz, index=False)
    if enviar_para_sheets:     enviar_linhas_para_sheet(aba_judge, buffer_sheet_judge, COLUNAS_JUDGE_SHEET)
    buffer_sheet_judge = []
    print(f"\n✅ {contador} novas avaliações nesta sessão. Total acumulado: {len(df_final)} / {len(elegiveis)}")
    if juiz_esgotado:
        print("Rode esta célula novamente para continuar de onde parou.")

    return df_final


In [ ]:
def rodar_experimento_com_checkpoint(
    df_textos: pd.DataFrame,
    modelos: list,
    caminho_checkpoint: str,
    delay_entre_chamadas: float = 0.1,
    salvar_a_cada: int = 20,
    enviar_para_sheets: bool = False,
) -> pd.DataFrame:

    df_checkpoint = carregar_checkpoint(caminho_checkpoint)

    # Conjunto de chaves já processadas, para consulta O(1)
    if len(df_checkpoint) > 0:
        chaves_feitas = set(
            zip(df_checkpoint["drug"], df_checkpoint["categoria"],
                df_checkpoint["modelo"], df_checkpoint["estrategia"])
        )
    else:
        chaves_feitas = set()

    modelos_esgotados_na_sessao = set()
    novos_resultados = []
    buffer_sheet_modelos = []
    contador_desde_ultimo_save = 0

    estrategias = [
        ("zero_shot", montar_prompt_zero_shot),
        ("few_shot", montar_prompt_few_shot),
    ]

    total_combinacoes = len(df_textos) * len(modelos) * len(estrategias)
    pendentes = total_combinacoes - len(chaves_feitas)
    print(f"Total de combinações: {total_combinacoes} | Já feitas: {len(chaves_feitas)} | Pendentes: {pendentes}")

    processadas_nesta_sessao = 0

    for _, row in df_textos.iterrows():
        for estrategia, montar_prompt in estrategias:
            prompt = montar_prompt(row["drug"], row["categoria"], row["texto"])

            for nome_modelo in modelos:
                chave = chave_execucao(row["drug"], row["categoria"], nome_modelo, estrategia)

                if chave in chaves_feitas:
                    continue  # já processado em sessão anterior

                if nome_modelo in modelos_esgotados_na_sessao:
                    continue  # já sabemos que esse modelo está sem cota hoje

                try:
                    resp = chamar_modelo(nome_modelo, prompt)
                    triplas = extrair_triplas_delimitadas(resp["texto_resposta"])
                    erro = None

                except CotaEsgotadaError as e:
                    print(f"⚠️  {nome_modelo}: cota esgotada nesta sessão. Pulando esse modelo até a próxima execução.")
                    modelos_esgotados_na_sessao.add(nome_modelo)
                    continue

                except Exception as e:
                    resp = {"texto_resposta": None, "tokens_prompt": None,
                             "tokens_resposta": None, "latencia_segundos": None}
                    triplas = None
                    erro = str(e)

                linha = {
                    "drug": row["drug"],
                    "categoria": row["categoria"],
                    "texto_fonte": row["texto"],
                    "modelo": nome_modelo,
                    "estrategia": estrategia,
                    "prompt": prompt,
                    "resposta_bruta": resp["texto_resposta"],
                    "triplas": triplas,
                    "n_triplas": len(triplas) if triplas else 0,
                    "parsing_ok": triplas is not None,
                        "predicado_categoria_ok": validar_predicado_categoria(row["categoria"], triplas),
                    "tokens_prompt": resp["tokens_prompt"],
                    "tokens_resposta": resp["tokens_resposta"],
                    "latencia_segundos": resp["latencia_segundos"],
                    "erro": erro,
                }
                novos_resultados.append(linha)
                buffer_sheet_modelos.append(linha)
                chaves_feitas.add(chave)
                processadas_nesta_sessao += 1
                contador_desde_ultimo_save += 1

                if processadas_nesta_sessao % 10 == 0:
                    print(f"[{processadas_nesta_sessao} processadas nesta sessão] {nome_modelo} | {estrategia} | {row['drug']} ({row['categoria']})")

                if contador_desde_ultimo_save >= salvar_a_cada:
                    df_atual = pd.concat([df_checkpoint, pd.DataFrame(novos_resultados)], ignore_index=True)
                    if enviar_para_sheets:                    enviar_linhas_para_sheet(aba_modelos, buffer_sheet_modelos, COLUNAS_MODELOS_SHEET)
                    buffer_sheet_modelos = []
                    salvar_checkpoint(df_atual, caminho_checkpoint)
                    contador_desde_ultimo_save = 0

                time.sleep(delay_entre_chamadas)

            # Se todos os modelos ficaram esgotados, não há motivo para continuar tentando.
            if len(modelos_esgotados_na_sessao) == len(modelos):
                print("🛑 Todos os modelos esgotaram a cota nesta sessão. Encerrando e salvando checkpoint.")
                df_final = pd.concat([df_checkpoint, pd.DataFrame(novos_resultados)], ignore_index=True)
                salvar_checkpoint(df_final, caminho_checkpoint)
                if enviar_para_sheets:enviar_linhas_para_sheet(aba_modelos, buffer_sheet_modelos, COLUNAS_MODELOS_SHEET)
                buffer_sheet_modelos = []
                print(f"Progresso salvo em: {caminho_checkpoint}")
                print("Rode esta célula novamente mais tarde para continuar de onde parou.")
                return df_final

    # Salvamento final ao concluir tudo (ou ao esgotar os itens do dataset)
    df_final = pd.concat([df_checkpoint, pd.DataFrame(novos_resultados)], ignore_index=True)
    salvar_checkpoint(df_final, caminho_checkpoint)
    if enviar_para_sheets: enviar_linhas_para_sheet(aba_modelos, buffer_sheet_modelos, COLUNAS_MODELOS_SHEET)
    buffer_sheet_modelos = []
    print(f"\n✅ Sessão concluída. {processadas_nesta_sessao} novas execuções processadas.")
    print(f"Total acumulado no checkpoint: {len(df_final)} / {total_combinacoes}")
    return df_final


In [ ]:
import os

CHECKPOINT_TESTE_PATH = os.path.join(PASTA_PROJETO, "checkpoint_TESTE_resultados.csv")
CHECKPOINT_TESTE_JUIZ_PATH = os.path.join(PASTA_PROJETO, "checkpoint_TESTE_avaliacao_juiz.csv")

for caminho in [CHECKPOINT_TESTE_PATH, CHECKPOINT_TESTE_JUIZ_PATH]:
    if os.path.exists(caminho):
        os.remove(caminho)
        print(f"Removido: {caminho}")

Removido: /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_TESTE_resultados.csv
Removido: /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_TESTE_avaliacao_juiz.csv


In [ ]:
N_AMOSTRAS_TESTE = 10  # ajuste livremente; 10-20 é suficiente para validar o pipeline

CHECKPOINT_TESTE_PATH = os.path.join(PASTA_PROJETO, "checkpoint_TESTE_resultados.csv")
CHECKPOINT_TESTE_JUIZ_PATH = os.path.join(PASTA_PROJETO, "checkpoint_TESTE_avaliacao_juiz.csv")

df_textos_teste = df_textos.sample(n=N_AMOSTRAS_TESTE, random_state=42).reset_index(drop=True)
print(f"Amostra de teste: {len(df_textos_teste)} textos")
df_textos_teste


Amostra de teste: 10 textos


,drug,categoria,texto,has_bioavailability,has_metabolism
0,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,False,True
1,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,False,True
2,Abiraterone,Metabolismo,Abiraterone acetate is hydrolyzed into active ...,False,True
3,Burosumab,Absorção,Burosumab absorption after subcutaneous inject...,False,False
4,Aranidipine,Metabolismo,Eight metabolites of aranidipine were found af...,False,True
5,Lisinopril,Absorção,Lisinopril is 6-60% orally bioavailable with a...,True,False
6,Bicisate,Metabolismo,Bicisate is metabolized to form mono- and di-a...,False,True
7,Pravastatin,Metabolismo,"After initial administration, pravastatin unde...",False,True
8,Paliperidone,Absorção,The absolute oral bioavailability of paliperid...,True,False
9,Mecamylamine,Absorção,Mecamylamine is almost completely absorbed fro...,False,False


In [ ]:
df_resultados_teste = rodar_experimento_com_checkpoint(
    df_textos_teste,
    modelos=list(MODELOS.keys()),
    caminho_checkpoint=CHECKPOINT_TESTE_PATH,
    delay_entre_chamadas=0.1,
    salvar_a_cada=20,
)

# Inspeção rápida: quantas execuções tiveram parsing válido, por modelo
print("\nTaxa de parsing válido por modelo (amostra de teste):")
print(df_resultados_teste.groupby("modelo")["parsing_ok"].mean())

df_resultados_teste.head(10)


Nenhum checkpoint encontrado em /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_TESTE_resultados.csv. Começando do zero.
Total de combinações: 60 | Já feitas: 0 | Pendentes: 60
[10 processadas nesta sessão] gpt-oss-120b | few_shot | Vigabatrin (Metabolismo)
[20 processadas nesta sessão] qwen-3.6-27b | zero_shot | Burosumab (Absorção)


/tmp/ipykernel_498/1483525875.py:93: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_atual = pd.concat([df_checkpoint, pd.DataFrame(novos_resultados)], ignore_index=True)


[30 processadas nesta sessão] llama-3.3-70b | few_shot | Aranidipine (Metabolismo)
⚠️  gpt-oss-120b: cota esgotada nesta sessão. Pulando esse modelo até a próxima execução.
[40 processadas nesta sessão] llama-3.3-70b | zero_shot | Pravastatin (Metabolismo)
[50 processadas nesta sessão] llama-3.3-70b | few_shot | Mecamylamine (Absorção)

✅ Sessão concluída. 50 novas execuções processadas.
Total acumulado no checkpoint: 50 / 60

Taxa de parsing válido por modelo (amostra de teste):
modelo
gpt-oss-120b     1.0
llama-3.3-70b    1.0
qwen-3.6-27b     1.0
Name: parsing_ok, dtype: object


/tmp/ipykernel_498/1483525875.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_checkpoint, pd.DataFrame(novos_resultados)], ignore_index=True)


,drug,categoria,texto_fonte,modelo,estrategia,prompt,resposta_bruta,triplas,n_triplas,parsing_ok,predicado_categoria_ok,tokens_prompt,tokens_resposta,latencia_segundos,erro
0,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,gpt-oss-120b,zero_shot,"Task: Extract knowledge triples (Subject, Pred...",Triclosan::metabolism::phase II metabolism via...,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,True,515,477,1.429,None
1,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,qwen-3.6-27b,zero_shot,"Task: Extract knowledge triples (Subject, Pred...",Triclosan::metabolism::phase II metabolism via...,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,True,483,38,0.319,None
2,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,llama-3.3-70b,zero_shot,"Task: Extract knowledge triples (Subject, Pred...",Triclosan::metabolism::phase II metabolism via...,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",3,True,True,486,51,0.504,None
3,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,gpt-oss-120b,few_shot,"Task: Extract knowledge triples (Subject, Pred...",Triclosan::metabolism::phase II metabolism via...,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,True,624,471,1.251,None
4,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,qwen-3.6-27b,few_shot,"Task: Extract knowledge triples (Subject, Pred...",Triclosan::metabolism::phase II metabolism via...,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,True,608,41,0.288,None
5,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,llama-3.3-70b,few_shot,"Task: Extract knowledge triples (Subject, Pred...",Triclosan::metabolism::phase II metabolism via...,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",3,True,True,599,52,0.428,None
6,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,gpt-oss-120b,zero_shot,"Task: Extract knowledge triples (Subject, Pred...",Vigabatrin::metabolism::not metabolized to any...,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",1,True,True,468,280,0.745,None
7,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,qwen-3.6-27b,zero_shot,"Task: Extract knowledge triples (Subject, Pred...",Vigabatrin::metabolism::not metabolized to any...,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",1,True,True,429,17,0.514,None
8,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,llama-3.3-70b,zero_shot,"Task: Extract knowledge triples (Subject, Pred...",Vigabatrin::metabolism::not metabolized to any...,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",2,True,True,438,32,0.401,None
9,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,gpt-oss-120b,few_shot,"Task: Extract knowledge triples (Subject, Pred...",Vigabatrin::metabolism::negligible,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",1,True,True,577,436,1.207,None


In [ ]:
df_avaliacoes_teste = avaliar_com_juiz_checkpoint(
    df_resultados_teste,
    caminho_checkpoint_juiz=CHECKPOINT_TESTE_JUIZ_PATH,
    delay_entre_chamadas=0.1,
    salvar_a_cada=20,
)

df_avaliacoes_teste.head(10)


Nenhum checkpoint do juiz encontrado. Começando do zero.
[10 avaliações nesta sessão] gpt-oss-120b | few_shot | Vigabatrin
[20 avaliações nesta sessão] qwen-3.6-27b | zero_shot | Burosumab
[30 avaliações nesta sessão] llama-3.3-70b | few_shot | Aranidipine
[40 avaliações nesta sessão] llama-3.3-70b | zero_shot | Pravastatin
[50 avaliações nesta sessão] llama-3.3-70b | few_shot | Mecamylamine

✅ 50 novas avaliações nesta sessão. Total acumulado: 50 / 50


,drug,categoria,modelo,estrategia,nota_format,nota_completeness,nota_correctness,nota_no_hallucination,justification,erro_juiz,nota_bioavailability
0,Triclosan,Metabolismo,gpt-oss-120b,zero_shot,5,4,5,5,Both triples are well-formed and accurately re...,None,None
1,Triclosan,Metabolismo,qwen-3.6-27b,zero_shot,5,5,5,5,Both triples are well-formed and semantically ...,None,None
2,Triclosan,Metabolismo,llama-3.3-70b,zero_shot,5,4,5,5,All triples are well-formed and semantically c...,None,None
3,Triclosan,Metabolismo,gpt-oss-120b,few_shot,5,4,5,5,Both triples are well-formed and accurately re...,None,None
4,Triclosan,Metabolismo,qwen-3.6-27b,few_shot,5,4,5,5,Both triples are well-formed and accurately re...,None,None
5,Triclosan,Metabolismo,llama-3.3-70b,few_shot,5,4,5,5,All triples are well-formed and semantically c...,None,None
6,Vigabatrin,Metabolismo,gpt-oss-120b,zero_shot,5,5,5,5,"The single extracted triple is well-formed, ca...",None,None
7,Vigabatrin,Metabolismo,qwen-3.6-27b,zero_shot,5,5,5,5,The single extracted triple is well-formed in ...,None,None
8,Vigabatrin,Metabolismo,llama-3.3-70b,zero_shot,5,5,4,5,Both triples are well-formed and capture the c...,None,None
9,Vigabatrin,Metabolismo,gpt-oss-120b,few_shot,5,4,5,5,The triple is well-formed with a clear subject...,None,None


In [ ]:
df_teste_completo = df_resultados_teste.merge(
    df_avaliacoes_teste,
    on=["drug", "categoria", "modelo", "estrategia"],
    how="left",
)

colunas_exportar = [
    "drug", "categoria", "texto_fonte", "modelo", "estrategia",
    "triplas", "n_triplas", "parsing_ok",
    "nota_format", "nota_completeness", "nota_correctness",
    "nota_no_hallucination", "justification",
]
df_teste_completo = df_teste_completo[colunas_exportar]

caminho_planilha_teste = os.path.join(PASTA_PROJETO, "amostra_teste_resultados.xlsx")
df_teste_completo.to_excel(caminho_planilha_teste, index=False)
print(f"Planilha exportada para: {caminho_planilha_teste}")

df_teste_completo

Planilha exportada para: /content/drive/MyDrive/PIBIC_grafos_conhecimento/amostra_teste_resultados.xlsx


,drug,categoria,texto_fonte,modelo,estrategia,triplas,n_triplas,parsing_ok,nota_format,nota_completeness,nota_correctness,nota_no_hallucination,justification
0,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,gpt-oss-120b,zero_shot,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,5,4,5,5,Both triples are well-formed and accurately re...
1,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,qwen-3.6-27b,zero_shot,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,5,5,5,5,Both triples are well-formed and semantically ...
2,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,llama-3.3-70b,zero_shot,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",3,True,5,4,5,5,All triples are well-formed and semantically c...
3,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,gpt-oss-120b,few_shot,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,5,4,5,5,Both triples are well-formed and accurately re...
4,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,qwen-3.6-27b,few_shot,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",2,True,5,4,5,5,Both triples are well-formed and accurately re...
5,Triclosan,Metabolismo,Triclosan is prone to phase II metabolism via ...,llama-3.3-70b,few_shot,"[{'sujeito': 'Triclosan', 'predicado': 'metabo...",3,True,5,4,5,5,All triples are well-formed and semantically c...
6,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,gpt-oss-120b,zero_shot,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",1,True,5,5,5,5,"The single extracted triple is well-formed, ca..."
7,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,qwen-3.6-27b,zero_shot,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",1,True,5,5,5,5,The single extracted triple is well-formed in ...
8,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,llama-3.3-70b,zero_shot,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",2,True,5,5,4,5,Both triples are well-formed and capture the c...
9,Vigabatrin,Metabolismo,Vigabatrin is not metabolized to any significa...,gpt-oss-120b,few_shot,"[{'sujeito': 'Vigabatrin', 'predicado': 'metab...",1,True,5,4,5,5,The triple is well-formed with a clear subject...


In [ ]:
[os.remove(_p) for _p in [os.path.join(PASTA_PROJETO, "checkpoint_VALIDACAO_PARSER.csv"), os.path.join(PASTA_PROJETO, "checkpoint_VALIDACAO_PARSER_juiz.csv")] if os.path.exists(_p)]
# Validação do parser corrigido com amostra NOVA (fora do checkpoint de teste existente)
df_textos_valida_novo = df_textos[~df_textos["drug"].isin(df_textos_teste["drug"])].sample(n=3, random_state=7)

CHECKPOINT_VALIDACAO_PATH = os.path.join(PASTA_PROJETO, "checkpoint_VALIDACAO_PARSER.csv")
CHECKPOINT_VALIDACAO_JUIZ_PATH = os.path.join(PASTA_PROJETO, "checkpoint_VALIDACAO_PARSER_juiz.csv")

df_resultados_valida = rodar_experimento_com_checkpoint(
    df_textos_valida_novo,
    modelos=list(MODELOS.keys()),
    caminho_checkpoint=CHECKPOINT_VALIDACAO_PATH,
    delay_entre_chamadas=0.1,
    salvar_a_cada=20,
)

print("\nContem tag de raciocinio na resposta bruta?")
print(df_resultados_valida["resposta_bruta"].str.contains("<think", case=False, na=False).value_counts())

print("\nTaxa de parsing valido por modelo:")
print(df_resultados_valida.groupby("modelo")["parsing_ok"].mean())

df_resultados_valida[["drug","modelo","estrategia","n_triplas","parsing_ok"]]

Nenhum checkpoint encontrado em /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_VALIDACAO_PARSER.csv. Começando do zero.
Total de combinações: 18 | Já feitas: 0 | Pendentes: 18
[10 processadas nesta sessão] gpt-oss-120b | few_shot | Sirolimus (Metabolismo)

✅ Sessão concluída. 18 novas execuções processadas.
Total acumulado no checkpoint: 18 / 18

Contem tag de raciocinio na resposta bruta?
resposta_bruta
False    18
Name: count, dtype: int64

Taxa de parsing valido por modelo:
modelo
gpt-oss-120b     1.0
llama-3.3-70b    1.0
qwen-3.6-27b     1.0
Name: parsing_ok, dtype: object


/tmp/ipykernel_498/1483525875.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_checkpoint, pd.DataFrame(novos_resultados)], ignore_index=True)


,drug,modelo,estrategia,n_triplas,parsing_ok
0,Zuclopenthixol,gpt-oss-120b,zero_shot,3,True
1,Zuclopenthixol,qwen-3.6-27b,zero_shot,2,True
2,Zuclopenthixol,llama-3.3-70b,zero_shot,4,True
3,Zuclopenthixol,gpt-oss-120b,few_shot,2,True
4,Zuclopenthixol,qwen-3.6-27b,few_shot,3,True
5,Zuclopenthixol,llama-3.3-70b,few_shot,5,True
6,Sirolimus,gpt-oss-120b,zero_shot,6,True
7,Sirolimus,qwen-3.6-27b,zero_shot,7,True
8,Sirolimus,llama-3.3-70b,zero_shot,4,True
9,Sirolimus,gpt-oss-120b,few_shot,10,True


In [25]:
df_resultados = rodar_experimento_com_checkpoint(
    df_textos,
    modelos=list(MODELOS.keys()),
    caminho_checkpoint=CHECKPOINT_PATH,
    delay_entre_chamadas=0.1,
    salvar_a_cada=20,
    enviar_para_sheets=True,
)

df_resultados.head()


Checkpoint carregado: 2250 execuções já registradas em /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_resultados.csv
Total de combinações: 25962 | Já feitas: 2250 | Pendentes: 23712
⚠️  llama-3.3-70b: cota esgotada nesta sessão. Pulando esse modelo até a próxima execução.
[10 processadas nesta sessão] gpt-oss-120b | zero_shot | Chlorotrianisene (Metabolismo)
[20 processadas nesta sessão] gpt-oss-120b | zero_shot | Topiramate (Absorção)
[30 processadas nesta sessão] gpt-oss-120b | zero_shot | Theophylline (Metabolismo)
[40 processadas nesta sessão] gpt-oss-120b | zero_shot | Liothyronine (Absorção)
⚠️  gpt-oss-120b: cota esgotada nesta sessão. Pulando esse modelo até a próxima execução.
[50 processadas nesta sessão] qwen-3.6-27b | zero_shot | Ardeparin (Absorção)
⚠️  qwen-3.6-27b: cota esgotada nesta sessão. Pulando esse modelo até a próxima execução.
🛑 Todos os modelos esgotaram a cota nesta sessão. Encerrando e salvando checkpoint.
Progresso salvo em: /content/drive/MyDri

,drug,categoria,texto_fonte,modelo,estrategia,prompt,resposta_bruta,triplas,n_triplas,parsing_ok,tokens_prompt,tokens_resposta,latencia_segundos,erro,predicado_categoria_ok
0,Lepirudin,Metabolismo,"As a polypeptide, lepirudin is expected to be ...",gpt-oss-120b,zero_shot,Tarefa: Extraia triplas de conhecimento (Sujei...,NaN,None,0,False,335,600,5.443992,NaN,NaN
1,Lepirudin,Metabolismo,"As a polypeptide, lepirudin is expected to be ...",gpt-oss-20b,zero_shot,Tarefa: Extraia triplas de conhecimento (Sujei...,NaN,None,0,False,335,600,0.756794,NaN,NaN
2,Lepirudin,Metabolismo,"As a polypeptide, lepirudin is expected to be ...",llama-3.3-70b,zero_shot,Tarefa: Extraia triplas de conhecimento (Sujei...,Lepirudin::é::um polipeptídeo\nLepirudin::é me...,"[{'sujeito': 'Lepirudin', 'predicado': 'é', 'o...",18,True,316,294,0.727387,NaN,NaN
3,Lepirudin,Metabolismo,"As a polypeptide, lepirudin is expected to be ...",maritaca-sabia-4,zero_shot,Tarefa: Extraia triplas de conhecimento (Sujei...,Lepirudin::é::um polipeptídeo\nLepirudin::é me...,"[{'sujeito': 'Lepirudin', 'predicado': 'é', 'o...",9,True,287,224,7.947766,NaN,NaN
4,Lepirudin,Metabolismo,"As a polypeptide, lepirudin is expected to be ...",gpt-oss-120b,few_shot,Tarefa: Extraia triplas de conhecimento (Sujei...,NaN,None,0,False,437,600,1.568307,NaN,NaN


In [ ]:
df_chk_check = carregar_checkpoint(CHECKPOINT_PATH)
total_check = len(df_textos) * len(MODELOS) * 2
print(f"Execucoes salvas no checkpoint agora: {len(df_chk_check)}")
print(f"Total de combinacoes do experimento: {total_check}")
print(f"Pendentes: {total_check - len(df_chk_check)}")
if "erro" in df_chk_check.columns:
    print(df_chk_check["erro"].dropna().value_counts().head(20))


Checkpoint carregado: 782 execuções já registradas em /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_resultados.csv
Execucoes salvas no checkpoint agora: 782
Total de combinacoes do experimento: 25962
Pendentes: 25180
Series([], Name: count, dtype: int64)


In [ ]:
        def calcular_metricas(df: pd.DataFrame) -> pd.DataFrame:
    metricas = df.groupby(["modelo", "estrategia"]).agg(
        n_execucoes=("drug", "count"),
        taxa_parsing_ok=("parsing_ok", "mean"),
        taxa_predicado_categoria_ok=("predicado_categoria_ok", "mean"),
        media_triplas_por_texto=("n_triplas", "mean"),
        tokens_prompt_medio=("tokens_prompt", "mean"),
        tokens_resposta_medio=("tokens_resposta", "mean"),
        latencia_media_segundos=("latencia_segundos", "mean"),
        taxa_erro=("erro", lambda x: x.notna().mean()),
    ).reset_index()

    metricas["tokens_total_medio"] = (
        metricas["tokens_prompt_medio"].fillna(0) + metricas["tokens_resposta_medio"].fillna(0)
    )

    return metricas.sort_values(["modelo", "estrategia"]).reset_index(drop=True)


df_metricas = calcular_metricas(df_resultados)
df_metricas


NameError: name 'pd' is not defined

In [ ]:
# ── Métricas de Tempo de Requisição e Tokens por Requisição ──────────────────

print("=" * 65)
print("  TEMPO DE REQUISIÇÃO (latência média por chamada, em segundos)")
print("=" * 65)
pivot_tempo = df_metricas.pivot(index="modelo", columns="estrategia", values="latencia_media_segundos")
pivot_tempo["média_geral"] = pivot_tempo.mean(axis=1)
pivot_tempo = pivot_tempo.sort_values("média_geral")
print(pivot_tempo.round(3).to_string())
print()

print("=" * 65)
print("  TOKENS POR REQUISIÇÃO (média de tokens de resposta gerados)")
print("=" * 65)
pivot_tokens_resp = df_metricas.pivot(index="modelo", columns="estrategia", values="tokens_resposta_medio")
pivot_tokens_resp["média_geral"] = pivot_tokens_resp.mean(axis=1)
pivot_tokens_resp = pivot_tokens_resp.sort_values("média_geral")
print(pivot_tokens_resp.round(1).to_string())
print()

print("=" * 65)
print("  TOKENS TOTAIS POR REQUISIÇÃO (prompt + resposta)")
print("=" * 65)
pivot_tokens_total = df_metricas.pivot(index="modelo", columns="estrategia", values="tokens_total_medio")
pivot_tokens_total["média_geral"] = pivot_tokens_total.mean(axis=1)
pivot_tokens_total = pivot_tokens_total.sort_values("média_geral")
print(pivot_tokens_total.round(1).to_string())
print()

print("=" * 65)
print("  TAXA DE PARSING VÁLIDO (% de respostas com triplas extraídas)")
print("=" * 65)
pivot_parsing = df_metricas.pivot(index="modelo", columns="estrategia", values="taxa_parsing_ok")
pivot_parsing["média_geral"] = pivot_parsing.mean(axis=1)
pivot_parsing = pivot_parsing.sort_values("média_geral", ascending=False)
print((pivot_parsing * 100).round(1).to_string() + "  (%)")


In [26]:
df_avaliacoes_juiz = avaliar_com_juiz_checkpoint(
    df_resultados,
    caminho_checkpoint_juiz=CHECKPOINT_JUIZ_PATH,
    delay_entre_chamadas=0.1,
    salvar_a_cada=20,
    enviar_para_sheets=True,
)


Checkpoint do juiz carregado: 2175 avaliações já feitas.
[10 avaliações nesta sessão] gpt-oss-120b | zero_shot | Chlorotrianisene
[20 avaliações nesta sessão] gpt-oss-120b | zero_shot | Topiramate
[30 avaliações nesta sessão] gpt-oss-120b | zero_shot | Theophylline
[40 avaliações nesta sessão] gpt-oss-120b | zero_shot | Liothyronine
[50 avaliações nesta sessão] qwen-3.6-27b | zero_shot | Ardeparin

✅ 55 novas avaliações nesta sessão. Total acumulado: 2230 / 2230


In [ ]:
df_extr_chk = carregar_checkpoint(CHECKPOINT_PATH)
df_juiz_chk = carregar_checkpoint(CHECKPOINT_JUIZ_PATH)
total_extracao = len(df_textos) * len(MODELOS) * 2
elegiveis_juiz = int(df_extr_chk["parsing_ok"].sum()) if "parsing_ok" in df_extr_chk.columns else 0
print(f"[EXTRACAO] feitas={len(df_extr_chk)} total={total_extracao} pendentes={total_extracao - len(df_extr_chk)}")
print(f"[JUIZ] feitas={len(df_juiz_chk)} elegiveis={elegiveis_juiz} pendentes={elegiveis_juiz - len(df_juiz_chk)}")

In [ ]:
df_extr_chk = carregar_checkpoint(CHECKPOINT_PATH)
df_juiz_chk = carregar_checkpoint(CHECKPOINT_JUIZ_PATH)
total_extracao = len(df_textos) * len(MODELOS) * 2
elegiveis_juiz = int(df_extr_chk["parsing_ok"].sum()) if "parsing_ok" in df_extr_chk.columns else 0
print(f"[EXTRACAO] feitas={len(df_extr_chk)} total={total_extracao} pendentes={total_extracao - len(df_extr_chk)}")
print(f"[JUIZ] feitas={len(df_juiz_chk)} elegiveis={elegiveis_juiz} pendentes={elegiveis_juiz - len(df_juiz_chk)}")

Checkpoint carregado: 1630 execuções já registradas em /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_resultados.csv
Checkpoint carregado: 1555 execuções já registradas em /content/drive/MyDrive/PIBIC_grafos_conhecimento/checkpoint_avaliacao_juiz.csv
[EXTRACAO] feitas=1630 total=25962 pendentes=24332
[JUIZ] feitas=1555 elegiveis=1555 pendentes=0


In [ ]:
# Médias das notas do juiz por modelo e estratégia
colunas_notas = ["nota_formato", "nota_completude", "nota_corretude", "nota_ausencia_alucinacao"]
df_notas_juiz = df_avaliacoes_juiz.groupby(["modelo", "estrategia"])[colunas_notas].mean().reset_index()
df_notas_juiz["nota_media_geral"] = df_notas_juiz[colunas_notas].mean(axis=1)
df_notas_juiz.sort_values("nota_media_geral", ascending=False)


In [ ]:
SEED = 42
FRACAO_AMOSTRA = 0.01

df_resultados_com_notas = df_resultados.merge(
    df_avaliacoes_juiz,
    on=["drug", "categoria", "modelo", "estrategia"],
    how="left",
)

elegiveis_para_amostra = df_resultados_com_notas[df_resultados_com_notas["parsing_ok"]].copy()

n_amostra_especialista = max(1, round(len(elegiveis_para_amostra) * FRACAO_AMOSTRA))

df_amostra_especialista = elegiveis_para_amostra.sample(
    n=n_amostra_especialista, random_state=SEED
).copy()

colunas_revisao = [
    "drug", "categoria", "texto_fonte", "modelo", "estrategia",
    "triplas", "nota_formato", "nota_completude", "nota_corretude",
    "nota_ausencia_alucinacao", "justificativa_juiz",
]
df_amostra_especialista = df_amostra_especialista[colunas_revisao]
df_amostra_especialista["avaliacao_especialista_1_5"] = ""
df_amostra_especialista["comentario_especialista"] = ""

caminho_amostra = os.path.join(PASTA_PROJETO, "amostra_revisao_especialista.xlsx")
df_amostra_especialista.to_excel(caminho_amostra, index=False)
print(f"Amostra de {len(df_amostra_especialista)} execuções exportada para: {caminho_amostra}")
df_amostra_especialista


In [ ]:
df_resumo_final = df_metricas.merge(df_notas_juiz, on=["modelo", "estrategia"], how="left")
df_resumo_final = df_resumo_final.sort_values("nota_media_geral", ascending=False).reset_index(drop=True)

caminho_resumo = os.path.join(PASTA_PROJETO, "resumo_final_comparativo.csv")
df_resumo_final.to_csv(caminho_resumo, index=False)
print(f"Resumo salvo em: {caminho_resumo}")
df_resumo_final


In [ ]:
from google.colab import data_table
data_table.DataTable(df_resumo_final, include_index=False, num_rows_per_page=10)
